In [4]:
import pandas as pd
import json
import random
from pathlib import Path

# --- CONFIGURATION ---
GENERATIONS_FOLDER = Path.cwd() / "generations"
REGIONS = ["output_montana", "output_new_mexico", "output_wyoming"] # Change loop logic if processing multiple regions
INPUT_DIRS = {region: GENERATIONS_FOLDER / region / "ra_diagnosed/csv" for region in REGIONS}

# Input Files
MEDS_FILES = {region: INPUT_DIRS[region] / "medications.csv" for region in REGIONS}
CONDITIONS_FILES = {region: INPUT_DIRS[region] / "conditions.csv" for region in REGIONS}
CANDIDATES_FILES = {region: f"adalimumab_candidates_{region}.json" for region in REGIONS} # Created by previous script

# Output
OUTPUT_PROMPTS_FILE = "all_clinical_prompts.json"

# --- CLINICAL CONSTANTS ---
# SNOMED Code for Rheumatoid Arthritis (To exclude from fluff/noise notes)
RA_CODE = 69896004 

# Failure Scenarios for the Target Note (Evidence for Prior Auth)
FAILURE_SCENARIOS = [
    {
        "type": "Lack of Efficacy",
        "detail": "Patient continues to have high disease activity (DAS28 > 5.1) with swollen joints despite compliance.",
        "plan": "Discontinue Methotrexate due to inefficacy. Initiate Adalimumab (Amjevita)."
    },
    {
        "type": "Hepatotoxicity",
        "detail": "Routine labs show elevated liver enzymes (ALT/AST > 3x upper limit) confirmed on re-test.",
        "plan": "Stop Methotrexate immediately due to liver toxicity. Switch to Adalimumab."
    },
    {
        "type": "GI Intolerance",
        "detail": "Patient reports severe nausea and vomiting post-dose, affecting quality of life.",
        "plan": "Methotrexate intolerance. Will start Adalimumab as next line therapy."
    }
]

def generate_target_prompt(patient_id, med_data):
    """
    Generates the 'Golden Evidence' note. This is the one the Agent MUST find.
    """
    scenario = random.choice(FAILURE_SCENARIOS)
    start_date = med_data['START']
    end_date = med_data['STOP'] if pd.notna(med_data['STOP']) else "TODAY"
    
    prompt = f"""
    ROLE: You are a Rheumatologist.
    TASK: Write a clinical SOAP note for a patient visit.
    
    PATIENT CONTEXT:
    - Patient ID: {patient_id}
    - Diagnosis: Rheumatoid Arthritis (Active)
    - Current Med History: Methotrexate (Started: {start_date}, Ended: {end_date})
    
    CLINICAL SCENARIO (MUST INCLUDE):
    The patient is failing Methotrexate due to: {scenario['type']}.
    Details: {scenario['detail']}
    
    INSTRUCTION:
    Write a professional medical note.
    CRITICAL: You must explicitly state that Methotrexate was taken from {start_date} to {end_date} and is being stopped due to {scenario['type']}.
    In the Plan section, formally request Prior Authorization for Adalimumab based on this failure.
    """
    return {
        "patient_id": patient_id,
        "type": "TARGET_EVIDENCE",
        "subtype": scenario['type'],
        "prompt": prompt
    }

def generate_fluff_prompts(patient_id, conditions_df, limit=3):
    """
    Generates 'Noise' notes based on the patient's other conditions.
    These exist to confuse simple keyword searches and prove the Agent's reasoning capabilities.
    """
    prompts = []
    
    # Filter conditions for this patient
    pat_conditions = conditions_df[conditions_df['PATIENT'] == patient_id]
    
    # EXCLUDE RA (Code 69896004) to avoid conflicting evidence
    fluff_conditions = pat_conditions[pat_conditions['CODE'] != RA_CODE]
    
    if fluff_conditions.empty:
        return []

    # Select random non-RA conditions
    selected_fluff = fluff_conditions.sample(n=min(len(fluff_conditions), limit))
    
    for _, row in selected_fluff.iterrows():
        condition_desc = row['DESCRIPTION']
        date = row['START']
        
        prompt = f"""
        ROLE: You are a General Practitioner or Specialist (NOT a Rheumatologist).
        TASK: Write a standard SOAP note for a routine visit.
        
        PATIENT CONTEXT:
        - Patient ID: {patient_id}
        - Date of Visit: {date}
        - Reason for Visit: {condition_desc}
        
        INSTRUCTION:
        Write a realistic, slightly messy clinical note focused ENTIRELY on '{condition_desc}'.
        Do NOT mention Rheumatoid Arthritis, Methotrexate, or Biologics.
        This is a background document unrelated to the current Prior Authorization request.
        """
        
        prompts.append({
            "patient_id": patient_id,
            "type": "FLUFF_NOISE",
            "subtype": condition_desc,
            "prompt": prompt
        })
        
    return prompts

def main():
    print(f"--- CLINICAL HISTORY GENERATOR ---")
    
    candidates = []
    for region in REGIONS:
        try:
            with open(CANDIDATES_FILES[region], 'r') as f:
                region_candidates = json.load(f)['READY_FOR_ADALIMUMAB_PA']
                print(f"Loaded {len(region_candidates)} candidates from {region}.")
                candidates.extend(region_candidates)
        except FileNotFoundError:
            print(f"Error: Could not find {CANDIDATES_FILES[region]}. Run the eligibility script for {region} first.")
            return

    # 2. Load Clinical Data (CSVs)
    print("Loading Synthea CSVs...")
    meds_df = pd.concat([pd.read_csv(MEDS_FILES[region]) for region in REGIONS], ignore_index=True)
    conds_df = pd.concat([pd.read_csv(CONDITIONS_FILES[region]) for region in REGIONS], ignore_index=True)

    all_prompts = []

    print("Generating prompts (Target + Fluff)...")
    for patient_id in candidates:
        
        # A. Generate TARGET Note (The Proof)
        # Find the Methotrexate entry
        pat_meds = meds_df[meds_df['PATIENT'] == patient_id]
        mtx_entry = pat_meds[pat_meds['DESCRIPTION'].str.contains('methotrexate', case=False, na=False)]
        
        if not mtx_entry.empty:
            # Use the last MTX prescription
            target_data = generate_target_prompt(patient_id, mtx_entry.iloc[-1])
            all_prompts.append(target_data)
        
        # B. Generate FLUFF Notes (The Noise)
        # Randomly generate 2 to 4 distractors per patient
        fluff_data = generate_fluff_prompts(patient_id, conds_df, limit=random.randint(2, 4))
        all_prompts.extend(fluff_data)

    # 3. Save Everything
    with open(OUTPUT_PROMPTS_FILE, "w") as f:
        json.dump(all_prompts, f, indent=2)

    print(f"\n[SUCCESS] Generated {len(all_prompts)} total prompts.")
    print(f" Breakdown:")
    print(f"  - Target Evidence Notes: {len([p for p in all_prompts if p['type'] == 'TARGET_EVIDENCE'])}")
    print(f"  - Fluff/Noise Notes:     {len([p for p in all_prompts if p['type'] == 'FLUFF_NOISE'])}")
    print(f"Saved to: {OUTPUT_PROMPTS_FILE}")
    print("Next Step: Run these prompts through MedGemma to create the .txt files.")

if __name__ == "__main__":
    main()

--- CLINICAL HISTORY GENERATOR ---
Loaded 139 candidates from output_montana.
Loaded 90 candidates from output_new_mexico.
Loaded 131 candidates from output_wyoming.
Loading Synthea CSVs...
Generating prompts (Target + Fluff)...

[SUCCESS] Generated 360 total prompts.
 Breakdown:
  - Target Evidence Notes: 360
  - Fluff/Noise Notes:     0
Saved to: all_clinical_prompts.json
Next Step: Run these prompts through MedGemma to create the .txt files.


In [5]:
import json
import requests
import time
import re
from pathlib import Path
from datetime import datetime

# --- CONFIGURATION ---
INPUT_FILE = "all_clinical_prompts.json"
OLLAMA_API_URL = "http://localhost:11434/api/generate"

# PHASE 1: VOLUME SETTINGS (Lightweight model for everyone)
MODEL_VOLUME = "MedAIBase/MedGemma1.5:4b"
DIR_VOLUME = Path("simulated_ehr_notes_all")

# PHASE 2: VIP SETTINGS (Heavy model for the demo subset)
MODEL_VIP = "puyangwang/medgemma-27b-it:q4_k_m"
DIR_VIP = Path("simulated_ehr_notes_vip")
VIP_PATIENT_LIMIT = 10  # Only generate high-quality notes for the first 10 patients

# --- UTILITIES ---

def sanitize_filename(text):
    """Cleans strings to be safe for filenames."""
    return re.sub(r'[^\w\-_]', '_', text)[:30]

def call_ollama(prompt, model, is_vip=False):
    """
    Generic function to call Ollama.
    Adjusts parameters based on whether it's a Volume run (fast) or VIP run (quality).
    """
    # VIP settings: Higher creativity, longer output, longer timeout
    if is_vip:
        params = {"num_predict": 600, "temperature": 0.7, "top_k": 40}
        timeout = 240 # 4 minutes max for 27B model
    # Volume settings: Deterministic, short output, fast timeout
    else:
        params = {"num_predict": 300, "temperature": 0.5}
        timeout = 45 # 45 seconds max

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": params
    }

    try:
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=timeout)
        response.raise_for_status()
        return response.json().get('response', '')
    except requests.exceptions.RequestException as e:
        print(f"   ⚠️ API Error: {e}")
        return None

def save_note(directory, patient_id, note_type, subtype, content):
    """Saves the generated content to a text file with a faux-EHR header."""
    filename = directory / f"Note_{patient_id}_{note_type}_{sanitize_filename(subtype)}.txt"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"PATIENT ID: {patient_id}\n")
        f.write(f"DOCUMENT TYPE: {note_type} - {subtype}\n")
        f.write(f"GENERATED DATE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"MODEL USED: {'VIP (27B)' if 'vip' in str(directory) else 'Volume (4B)'}\n")
        f.write("-" * 50 + "\n\n")
        f.write(content)
    return filename

# --- PHASE 1: VOLUME GENERATION ---

def run_volume_generation(prompts_data):
    print(f"\n{'='*60}")
    print(f"🚀 PHASE 1: VOLUME GENERATION (All Patients)")
    print(f"🤖 Model: {MODEL_VOLUME} (Optimized for speed)")
    print(f"📂 Output: {DIR_VOLUME}")
    print(f"{'='*60}")

    DIR_VOLUME.mkdir(parents=True, exist_ok=True)
    success_count = 0
    start_time = time.time()

    for i, data in enumerate(prompts_data):
        pid = data['patient_id']
        note_type = "TARGET" if data['type'] == "TARGET_EVIDENCE" else "FLUFF"
        subtype = data.get('subtype', 'General')
        
        # Check if file exists to skip (Idempotency)
        fname = DIR_VOLUME / f"Note_{pid}_{note_type}_{sanitize_filename(subtype)}.txt"
        if fname.exists():
            continue

        # Generate
        content = call_ollama(data['prompt'], MODEL_VOLUME, is_vip=False)
        
        if content:
            save_note(DIR_VOLUME, pid, note_type, subtype, content)
            success_count += 1
            # Simple progress log every 10 items to reduce spam
            if i % 10 == 0:
                print(f"[{i}/{len(prompts_data)}] Progress... ({time.time() - start_time:.0f}s elapsed)")

    print(f"✅ PHASE 1 COMPLETE: {success_count} notes generated in {DIR_VOLUME}")

# --- PHASE 2: VIP GENERATION ---

def run_vip_generation(prompts_data):
    print(f"\n{'='*60}")
    print(f"💎 PHASE 2: VIP DEMO GENERATION (Top {VIP_PATIENT_LIMIT} Patients)")
    print(f"🧠 Model: {MODEL_VIP} (Optimized for medical accuracy)")
    print(f"📂 Output: {DIR_VIP}")
    print(f"{'='*60}")

    DIR_VIP.mkdir(parents=True, exist_ok=True)

    # Filter prompts to only include the first N unique patient IDs
    unique_ids = list(set(p['patient_id'] for p in prompts_data))[:VIP_PATIENT_LIMIT]
    vip_prompts = [p for p in prompts_data if p['patient_id'] in unique_ids]

    print(f"📋 Selected {len(unique_ids)} VIP patients ({len(vip_prompts)} total notes to write)...\n")

    for i, data in enumerate(vip_prompts):
        pid = data['patient_id']
        note_type = "TARGET" if data['type'] == "TARGET_EVIDENCE" else "FLUFF"
        subtype = data.get('subtype', 'General')

        fname = DIR_VIP / f"Note_{pid}_{note_type}_{sanitize_filename(subtype)}.txt"
        if fname.exists():
            print(f"⏭️  Skipping existing: {pid} - {subtype}")
            continue

        print(f"[{i+1}/{len(vip_prompts)}] Generating VIP Note: {pid} | {subtype}...")
        start_task = time.time()
        
        content = call_ollama(data['prompt'], MODEL_VIP, is_vip=True)
        
        if content:
            save_note(DIR_VIP, pid, note_type, subtype, content)
            duration = time.time() - start_task
            print(f"   ✅ Done in {duration:.1f}s")
        else:
            print("   ❌ Failed to generate.")

    print(f"✨ PHASE 2 COMPLETE. Your high-quality demo data is in {DIR_VIP}")

# --- MAIN ENTRY POINT ---

if __name__ == "__main__":
    if not Path(INPUT_FILE).exists():
        print(f"❌ Error: Input file '{INPUT_FILE}' not found.")
        print("   -> Please run the 'Clinical History Generator' script first.")
    else:
        # Load prompts once
        with open(INPUT_FILE, 'r') as f:
            all_prompts = json.load(f)
        
        # Execute Phase 1 (Volume)
        # Note: You can comment this out if you only want to run the VIP batch
        run_volume_generation(all_prompts)
        
        # Execute Phase 2 (VIP/Quality)
        run_vip_generation(all_prompts)


🚀 PHASE 1: VOLUME GENERATION (All Patients)
🤖 Model: MedAIBase/MedGemma1.5:4b (Optimized for speed)
📂 Output: simulated_ehr_notes_all
[0/360] Progress... (12s elapsed)
[10/360] Progress... (115s elapsed)
[20/360] Progress... (226s elapsed)
[30/360] Progress... (333s elapsed)
[40/360] Progress... (432s elapsed)
[50/360] Progress... (530s elapsed)
[60/360] Progress... (628s elapsed)
[70/360] Progress... (727s elapsed)
[80/360] Progress... (825s elapsed)
[90/360] Progress... (924s elapsed)
[100/360] Progress... (1023s elapsed)
[110/360] Progress... (1122s elapsed)
[120/360] Progress... (1220s elapsed)
[130/360] Progress... (1319s elapsed)
[140/360] Progress... (1420s elapsed)
[150/360] Progress... (1520s elapsed)
[160/360] Progress... (1621s elapsed)
[170/360] Progress... (1720s elapsed)
[180/360] Progress... (1825s elapsed)
[190/360] Progress... (1924s elapsed)
[200/360] Progress... (2023s elapsed)
[210/360] Progress... (2125s elapsed)


KeyboardInterrupt: 

In [ ]:
import json
import re
from pathlib import Path

# --- CONFIGURATION ---
DIRS_TO_INDEX = [
    Path("simulated_ehr_notes_vip"),
    Path("simulated_ehr_notes_all")
]

OUTPUT_FILE = "patient_manifest.json"

def parse_filename(filename):
    """
    Extrait les infos du nom de fichier : Note_{UUID}_{TYPE}_{SUBTYPE}.txt
    """
    # Regex pour capturer l'UUID (qui contient des tirets) et le reste
    # Structure attendue : Note_uuid-uuid_TYPE_Subtype.txt
    match = re.match(r"Note_([a-z0-9\-]+)_(TARGET|FLUFF)_(.*)\.txt", filename)
    
    if match:
        return {
            "patient_id": match.group(1),
            "type": match.group(2), # TARGET ou FLUFF
            "subtype": match.group(3).replace("_", " "),
            "filename": filename
        }
    return None

def main():
    manifest = {}
    
    print(f"🕵️‍♂️ Démarrage de l'indexation...")

    for directory in DIRS_TO_INDEX:
        if not directory.exists():
            print(f"⚠️ Dossier introuvable (ignoré) : {directory}")
            continue

        print(f"📂 Scanning : {directory} ...")
        
        # On liste tous les fichiers .txt
        files = sorted(list(directory.glob("*.txt")))
        
        for file_path in files:
            info = parse_filename(file_path.name)
            
            if info:
                pid = info['patient_id']
                
                # Initialiser l'entrée patient si elle n'existe pas
                if pid not in manifest:
                    manifest[pid] = {
                        "patient_id": pid,
                        "docs": [],
                        "has_target": False, # Utile pour filtrer dans l'UI
                        "source_dir": str(directory)
                    }
                
                # Ajouter le document
                manifest[pid]["docs"].append({
                    "path": str(file_path),
                    "type": info['type'],
                    "description": info['subtype']
                })
                
                # Taguer si c'est un dossier "Gagnant" (avec preuve)
                if info['type'] == "TARGET":
                    manifest[pid]["has_target"] = True

    # Sauvegarde du JSON
    with open(OUTPUT_FILE, "w") as f:
        json.dump(manifest, f, indent=2)

    total_patients = len(manifest)
    total_docs = sum(len(p['docs']) for p in manifest.values())

    print(f"\n{'='*50}")
    print(f"✅ MANIFESTE GÉNÉRÉ : {OUTPUT_FILE}")
    print(f"👥 Patients indexés : {total_patients}")
    print(f"📄 Documents liés   : {total_docs}")
    print(f"{'='*50}")

if __name__ == "__main__":
    main()